In [2]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# =====================================================
# 1. ĐỌC DỮ LIỆU
# =====================================================

INPUT_FILE = ROOT / "data" / "clean" / "Chuan_hoa_Toan_Tinh.xlsx"
OUTPUT_FILE = ROOT / "data" / "model" / "household_featured.xlsx"
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Du_lieu_chuan_hoa"
)

df = df.reset_index(drop=True).copy()
df["SOURCE_ROW"] = df.index + 1


def clean_text(s):
    return s.astype(str).str.strip().replace({"nan": "", "None": ""})


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value).strip())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.upper()


for col in ["STT_HO", "HUYEN", "XA", "HO_TEN_CHU_HO", "QUANHE_CHUHO", "GIOI_TINH", "DAN_TOC"]:
    if col in df.columns:
        df[col] = clean_text(df[col])

df["QUANHE_NORM"] = df["QUANHE_CHUHO"].map(normalize_text)
df["GIOI_TINH_NORM"] = df["GIOI_TINH"].map(normalize_text)

# =====================================================
# 2. TẠO KHÓA HỘ SẠCH
# =====================================================
# Không group trực tiếp theo STT_HO vì STT_HO có đoạn bị gom sai.
# Tách hộ khi một trong các khóa chính thay đổi.
# QUANHE_CHUHO chỉ dùng làm cờ kiểm tra, không dùng để tách hộ.

household_key_cols = ["STT_HO", "HUYEN", "XA", "HO_TEN_CHU_HO"]

is_new_household = pd.Series(False, index=df.index)

for col in household_key_cols:
    is_new_household = is_new_household | df[col].ne(df[col].shift())

is_new_household.iloc[0] = True

df["HOUSEHOLD_ID"] = is_new_household.cumsum().astype("int64")
df["NHAN_KHAU_GOC"] = pd.to_numeric(df["NHAN_KHAU"], errors="coerce")

# =====================================================
# 3. FEATURE ENGINEERING NHÓM TUỔI VÀ CỜ ĐẾM
# =====================================================

df["TUOI"] = pd.to_numeric(df["TUOI"], errors="coerce")

is_chu_ho = df["QUANHE_NORM"].eq("CHU HO")

df["IS_CHU_HO"] = is_chu_ho.astype(int)
df["IS_NAM"] = df["GIOI_TINH_NORM"].eq("NAM").astype(int)
df["IS_NU"] = df["GIOI_TINH_NORM"].eq("NU").astype(int)

df["MAM_NON"] = df["TUOI"].between(0, 5).astype(int)
df["HOC_DUONG"] = df["TUOI"].between(6, 17).astype(int)
df["LAO_DONG"] = df["TUOI"].between(18, 49).astype(int)
df["U50"] = (df["TUOI"] >= 50).astype(int)

# =====================================================
# 4. THÔNG TIN CHỦ HỘ
# =====================================================

chu_ho = df[is_chu_ho].copy()
chu_ho = chu_ho.sort_values("SOURCE_ROW").drop_duplicates("HOUSEHOLD_ID", keep="first")

bins = [0, 34, 49, 64, 120]
labels = ["<35", "35-49", "50-64", "65+"]

chu_ho["NHOM_TUOI_CHU_HO"] = pd.cut(
    chu_ho["TUOI"],
    bins=bins,
    labels=labels
)

chu_ho = chu_ho[
    [
        "HOUSEHOLD_ID",
        "HO_TEN_CHU_HO",
        "TUOI",
        "GIOI_TINH",
        "DAN_TOC",
        "NHOM_TUOI_CHU_HO"
    ]
].rename(
    columns={
        "TUOI": "TUOI_CHU_HO",
        "GIOI_TINH": "GIOI_TINH_CHU_HO",
        "DAN_TOC": "DAN_TOC_CHU_HO"
    }
)

# =====================================================
# 5. TỔNG HỢP CẤP HỘ
# =====================================================

household = (
    df.groupby("HOUSEHOLD_ID", sort=False)
      .agg(
          STT_HO=("STT_HO", "first"),
          HUYEN=("HUYEN", "first"),
          XA=("XA", "first"),
          PHAN_LOAI_HO=("PHAN_LOAI_HO", "first"),
          HO_TEN_CHU_HO_FALLBACK=("HO_TEN_CHU_HO", "first"),

          NHAN_KHAU=("HOUSEHOLD_ID", "size"),
          NHAN_KHAU_GOC_MAX=("NHAN_KHAU_GOC", "max"),

          SO_CHU_HO=("IS_CHU_HO", "sum"),
          SO_XA_TRONG_HO=("XA", "nunique"),
          SO_HUYEN_TRONG_HO=("HUYEN", "nunique"),
          SOURCE_ROW_MIN=("SOURCE_ROW", "min"),
          SOURCE_ROW_MAX=("SOURCE_ROW", "max"),

          SO_NAM=("IS_NAM", "sum"),
          SO_NU=("IS_NU", "sum"),
          SO_MAM_NON=("MAM_NON", "sum"),
          SO_HOC_DUONG=("HOC_DUONG", "sum"),
          SO_LAO_DONG=("LAO_DONG", "sum"),
          SO_U50=("U50", "sum")
      )
      .reset_index()
)

# =====================================================
# 6. TÍNH CÁC CHỈ SỐ PHỤ THUỘC
# =====================================================

household["CO_DONG_CHU_HO"] = (household["SO_CHU_HO"] > 0).astype(int)

household["SO_PHU_THUOC"] = (
    household["SO_MAM_NON"]
    + household["SO_HOC_DUONG"]
    + household["SO_U50"]
)

household["TY_LE_MAM_NON"] = household["SO_MAM_NON"] / household["NHAN_KHAU"]
household["TY_LE_HOC_DUONG"] = household["SO_HOC_DUONG"] / household["NHAN_KHAU"]
household["TY_LE_LAO_DONG"] = household["SO_LAO_DONG"] / household["NHAN_KHAU"]
household["TY_LE_U50"] = household["SO_U50"] / household["NHAN_KHAU"]

household["TY_LE_PHU_THUOC"] = (
    household["SO_PHU_THUOC"]
    / household["SO_LAO_DONG"].replace(0, np.nan)
)

# =====================================================
# 7. THỐNG KÊ TUỔI HỘ
# =====================================================

age_stats = (
    df.groupby("HOUSEHOLD_ID", sort=False)
      .agg(
          TUOI_TRUNG_BINH_HO=("TUOI", "mean"),
          TUOI_LON_NHAT=("TUOI", "max"),
          TUOI_NHO_NHAT=("TUOI", "min")
      )
      .reset_index()
)

age_stats["DO_LECH_TUOI"] = (
    age_stats["TUOI_LON_NHAT"]
    - age_stats["TUOI_NHO_NHAT"]
)

# =====================================================
# 8. GHÉP THÔNG TIN CHỦ HỘ VÀ XUẤT FILE
# =====================================================

household_featured = household.merge(
    chu_ho,
    on="HOUSEHOLD_ID",
    how="left"
)

household_featured = household_featured.merge(
    age_stats,
    on="HOUSEHOLD_ID",
    how="left"
)

household_featured["HO_TEN_CHU_HO"] = household_featured["HO_TEN_CHU_HO"].combine_first(
    household_featured["HO_TEN_CHU_HO_FALLBACK"]
)

household_featured = household_featured.drop(columns=["HO_TEN_CHU_HO_FALLBACK"])

ordered_cols = [
    "HOUSEHOLD_ID",
    "STT_HO",
    "HUYEN",
    "XA",
    "PHAN_LOAI_HO",
    "NHAN_KHAU",
    "NHAN_KHAU_GOC_MAX",
    "SO_CHU_HO",
    "SO_XA_TRONG_HO",
    "SO_HUYEN_TRONG_HO",
    "CO_DONG_CHU_HO",
    "SOURCE_ROW_MIN",
    "SOURCE_ROW_MAX",
    "SO_NAM",
    "SO_NU",
    "SO_MAM_NON",
    "SO_HOC_DUONG",
    "SO_LAO_DONG",
    "SO_U50",
    "SO_PHU_THUOC",
    "TY_LE_MAM_NON",
    "TY_LE_HOC_DUONG",
    "TY_LE_LAO_DONG",
    "TY_LE_U50",
    "TY_LE_PHU_THUOC",
    "HO_TEN_CHU_HO",
    "TUOI_CHU_HO",
    "GIOI_TINH_CHU_HO",
    "DAN_TOC_CHU_HO",
    "NHOM_TUOI_CHU_HO",
    "TUOI_TRUNG_BINH_HO",
    "TUOI_LON_NHAT",
    "TUOI_NHO_NHAT",
    "DO_LECH_TUOI"
]

household_featured = household_featured[
    [c for c in ordered_cols if c in household_featured.columns]
]

print("Số hộ:", f"{household_featured.shape[0]:,}")
print("Số biến:", household_featured.shape[1])
print("NHAN_KHAU max sau sửa:", household_featured["NHAN_KHAU"].max())
print("Số hộ có nhiều hơn 1 chủ hộ:", int((household_featured["SO_CHU_HO"] > 1).sum()))
print("Số hộ trải qua nhiều xã:", int((household_featured["SO_XA_TRONG_HO"] > 1).sum()))
print("Số hộ trải qua nhiều huyện:", int((household_featured["SO_HUYEN_TRONG_HO"] > 1).sum()))

display(household_featured.head())
display(household_featured.sort_values("NHAN_KHAU", ascending=False).head(20))

household_featured.to_excel(OUTPUT_FILE, index=False)

print("Đã lưu:", OUTPUT_FILE)

Số hộ: 26,953
Số biến: 34
NHAN_KHAU max sau sửa: 21
Số hộ có nhiều hơn 1 chủ hộ: 0
Số hộ trải qua nhiều xã: 0
Số hộ trải qua nhiều huyện: 0


,HOUSEHOLD_ID,STT_HO,HUYEN,XA,PHAN_LOAI_HO,NHAN_KHAU,NHAN_KHAU_GOC_MAX,SO_CHU_HO,SO_XA_TRONG_HO,SO_HUYEN_TRONG_HO,...,TY_LE_PHU_THUOC,HO_TEN_CHU_HO,TUOI_CHU_HO,GIOI_TINH_CHU_HO,DAN_TOC_CHU_HO,NHOM_TUOI_CHU_HO,TUOI_TRUNG_BINH_HO,TUOI_LON_NHAT,TUOI_NHO_NHAT,DO_LECH_TUOI
0,1,1,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,1,1.0,1,1,1,...,NaN,TRƯƠNG THỊ LÙN,87.0,Nữ,Kinh,65+,87.000000,87.0,87.0,0.0
1,2,2,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,4,4.0,1,1,1,...,1.0,HOÀNG VĂN TRƯỜNG,46.0,Nam,Kinh,35-49,29.750000,46.0,13.0,33.0
2,3,3,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,2,2.0,1,1,1,...,NaN,TRẦN VĂN LONG,66.0,Nam,Kinh,65+,60.500000,66.0,55.0,11.0
3,4,4,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,1,1.0,1,1,1,...,NaN,NGUYỄN THỊ TƯƠI,80.0,Nữ,Kinh,65+,80.000000,80.0,80.0,0.0
4,5,5,Huyện Cư Jút,TÂM THẮNG,Hộ nghèo,6,6.0,1,1,1,...,5.0,ĐỖ VĂN PHÚC,66.0,Nam,Kinh,65+,33.333333,66.0,10.0,56.0


,HOUSEHOLD_ID,STT_HO,HUYEN,XA,PHAN_LOAI_HO,NHAN_KHAU,NHAN_KHAU_GOC_MAX,SO_CHU_HO,SO_XA_TRONG_HO,SO_HUYEN_TRONG_HO,...,TY_LE_PHU_THUOC,HO_TEN_CHU_HO,TUOI_CHU_HO,GIOI_TINH_CHU_HO,DAN_TOC_CHU_HO,NHOM_TUOI_CHU_HO,TUOI_TRUNG_BINH_HO,TUOI_LON_NHAT,TUOI_NHO_NHAT,DO_LECH_TUOI
25933,25934,21101,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,21,NaN,0,1,1,...,1.625000,LÝ GIỐNG DIA,NaN,NaN,NaN,NaN,18.428571,50.0,5.0,45.0
18112,18113,16078,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,18,18.0,1,1,1,...,1.250000,HẠNG A SA,30.0,Nam,Dao,<35,18.944444,61.0,4.0,57.0
4469,4470,4280,Huyện Đắk Glong,ĐẮK R'MĂNG,Hộ nghèo,17,17.0,1,1,1,...,1.666667,GIÀNG SEO SỲ,55.0,Nam,Dao,50-64,21.437500,55.0,5.0,50.0
25947,25948,21105,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,17,NaN,0,1,1,...,0.888889,SÌNH CHẮNG PHỒNG,NaN,NaN,NaN,NaN,19.764706,50.0,4.0,46.0
4199,4200,4025,Huyện Đắk Glong,ĐẮK R'MĂNG,Hộ nghèo,16,16.0,1,1,1,...,2.200000,LẦU A DUA,34.0,Nam,Dao,<35,18.687500,49.0,6.0,43.0
25963,25964,21107,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,16,16.0,1,1,1,...,0.777778,SÙNG A TÙNG,59.0,Nam,Dao,50-64,25.500000,59.0,7.0,52.0
17804,17805,15777,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,15,15.0,1,1,1,...,1.142857,SÙNG PHẮNG DẾ,62.0,Nam,Dao,50-64,24.466667,62.0,4.0,58.0
439,440,436,Huyện Cư Jút,ĐẮK D'RÔNG,Hộ nghèo,15,15.0,1,1,1,...,1.500000,LÝ A PÁO,57.0,Nam,Dao,50-64,28.000000,97.0,6.0,91.0
6792,6793,6500,Huyện Đắk Glong,ĐẮK SOM,Hộ nghèo,15,15.0,1,1,1,...,1.000000,SÌNH A PHÚ,48.0,Nam,Dao,35-49,19.714286,48.0,4.0,44.0
18138,18139,16104,Huyện Tuy Đức,QUẢNG TRỰC,Hộ nghèo,15,15.0,1,1,1,...,0.875000,SÙNG A DƠ,53.0,Nam,Dao,50-64,26.200000,58.0,5.0,53.0


Đã lưu: C:\NguyenHoangThanhTam\data\model\household_featured.xlsx
